# VAE — Cat Image Generation
Variational Autoencoder trained on the preprocessed cat dataset.

In [ ]:
import subprocess
subprocess.run(["git", "clone", "https://github.com/sienkozuzanna/generative-models"], check=True)

import sys
sys.path.append("generative-models")
print("Repo cloned successfully")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from src.data.dataset import get_dataloader, denormalize
from src.models.vae import VAE, vae_loss

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
CONFIG = {
    # data
    "data_dir" : "/kaggle/input/cats-processed-64/cats_64",
    "image_size": 64,
    "batch_size": 64,

    # model
    "latent_dim": 128,
    "base_channels": 32,

    # training
    "epochs": 50,
    "lr": 1e-3,
    "beta": 1.0,

    # saving
    "checkpoint_dir": "/kaggle/working/vae_outputs",
    "sample_interval": 5
}

Path(CONFIG["checkpoint_dir"]).mkdir(exist_ok=True)
print(CONFIG)

In [ ]:
loader = get_dataloader(CONFIG["data_dir"], image_size=CONFIG["image_size"], batch_size=CONFIG["batch_size"],
    augment=True)

print(f"Images: {len(loader.dataset)}")
print(f"Batches/epoch: {len(loader)}")

batch = next(iter(loader))
imgs = denormalize(batch[:16]).permute(0, 2, 3, 1).numpy()
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for ax, img in zip(axes.flatten(), imgs):
    ax.imshow(img)
    ax.axis("off")
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()

In [ ]:
model = VAE(image_size=CONFIG["image_size"], latent_dim=CONFIG["latent_dim"], 
            base_channels=CONFIG["base_channels"]).to(device)

optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {params:,}")

In [ ]:
def save_samples(model, epoch, n=16, device=device):
    """Generate and display sample images."""
    model.eval()
    samples = model.generate(n, device)
    imgs = denormalize(samples.cpu()).permute(0, 2, 3, 1).numpy()

    fig, axes = plt.subplots(2, n // 2, figsize=(16, 5))
    for ax, img in zip(axes.flatten(), imgs):
        ax.imshow(img.clip(0, 1))
        ax.axis("off")
    plt.suptitle(f"Generated samples — epoch {epoch}")
    plt.tight_layout()
    plt.savefig(f"{CONFIG['checkpoint_dir']}/samples_epoch_{epoch:03d}.png",
                dpi=100, bbox_inches='tight')
    plt.show()
    model.train()

# fixed noise for consistent comparison across epochs
fixed_z = torch.randn(16, CONFIG["latent_dim"]).to(device)

In [ ]:
history = {"total": [], "recon": [], "kl": []}
best_loss = float("inf")

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    total_loss = recon_loss_sum = kl_loss_sum = 0.0

    for batch in loader:
        x = batch.to(device)

        optimizer.zero_grad()
        recon, mu, log_var = model(x)
        loss, rl, kl = vae_loss(recon, x, mu, log_var, beta=CONFIG["beta"])
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        recon_loss_sum += rl.item()
        kl_loss_sum += kl.item()

    n = len(loader)
    avg_total = total_loss / n
    avg_recon = recon_loss_sum / n
    avg_kl = kl_loss_sum / n

    history["total"].append(avg_total)
    history["recon"].append(avg_recon)
    history["kl"].append(avg_kl)

    scheduler.step(avg_total)

    print(f"Epoch [{epoch:3d}/{CONFIG['epochs']}] "
          f"Loss: {avg_total:.4f} "
          f"(recon={avg_recon:.4f}, kl={avg_kl:.4f})")

    # save best model
    if avg_total < best_loss:
        best_loss = avg_total
        torch.save(model.state_dict(),
                   f"{CONFIG['checkpoint_dir']}/vae_best.pt")

    # save samples every N epochs
    if epoch % CONFIG["sample_interval"] == 0:
        save_samples(model, epoch)

print(f"\nTraining done. Best loss: {best_loss:.4f}")

Loss curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, key, color in zip(axes, ["total", "recon", "kl"], ["steelblue", "seagreen", "tomato"]):
    ax.plot(history[key], color=color)
    ax.set_title(f"{key.capitalize()} loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.3)

plt.suptitle("VAE Training Curves")
plt.tight_layout()
plt.savefig(f"{CONFIG['checkpoint_dir']}/loss_curves.png", dpi=100, bbox_inches='tight')
plt.show()

Reconstruction quality

Compare original images with their reconstructions.

In [ ]:
# load best model
model.load_state_dict(torch.load(f"{CONFIG['checkpoint_dir']}/vae_best.pt"))
model.eval()

with torch.no_grad():
    test_batch = next(iter(loader))[:8].to(device)
    recon, _, _ = model(test_batch)

originals = denormalize(test_batch.cpu()).permute(0, 2, 3, 1).numpy()
reconstructed = denormalize(recon.cpu()).permute(0, 2, 3, 1).numpy()

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i in range(8):
    axes[0, i].imshow(originals[i].clip(0, 1))
    axes[0, i].axis("off")
    axes[1, i].imshow(reconstructed[i].clip(0, 1))
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("Reconstructed", fontsize=12)
plt.suptitle("Reconstruction Quality")
plt.tight_layout()
plt.savefig(f"{CONFIG['checkpoint_dir']}/reconstructions.png", dpi=100, bbox_inches='tight')
plt.show()

Latent space interpolation

Select 2 generated images and interpolate between their latent vectors.

In [ ]:
model.eval()

# encode two real images to get their latent vectors
with torch.no_grad():
    imgs_for_interp = next(iter(loader))[:2].to(device)
    z1 = model.encode(imgs_for_interp[0:1]) # [1, latent_dim]
    z2 = model.encode(imgs_for_interp[1:2]) # [1, latent_dim]

# interpolate: 8 intermediate + 2 endpoints = 10 images total
interp_images = model.interpolate(z1[0], z2[0], steps=8)  # [10, 3, H, W]
interp_imgs = denormalize(interp_images.cpu()).permute(0, 2, 3, 1).numpy()

fig, axes = plt.subplots(1, 10, figsize=(20, 3))
labels = ["z₁"] + [f"t={i/9:.1f}" for i in range(1, 9)] + ["z₂"]

for ax, img, label in zip(axes, interp_imgs, labels):
    ax.imshow(img.clip(0, 1))
    ax.set_title(label, fontsize=8)
    ax.axis("off")

plt.suptitle("Latent Space Interpolation (z₁ → z₂)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{CONFIG['checkpoint_dir']}/interpolation.png", dpi=100, bbox_inches='tight')
plt.show()

print("Interpolation saved.")
print("For the report: z1 and z2 are the two endpoint images,")
print("the 8 middle images are generated by linear interpolation.")

Final generated samples

In [ ]:
model.eval()
samples = model.generate(32, device)
imgs = denormalize(samples.cpu()).permute(0, 2, 3, 1).numpy()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for ax, img in zip(axes.flatten(), imgs):
    ax.imshow(img.clip(0, 1))
    ax.axis("off")
plt.suptitle("VAE — Generated Cat Images", fontsize=14)
plt.tight_layout()
plt.savefig(f"{CONFIG['checkpoint_dir']}/final_samples.png", dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
print("Saved files:")
for f in sorted(Path(CONFIG["checkpoint_dir"]).rglob("*")):
    size_kb = f.stat().st_size / 1024
    print(f" {f.name} ({size_kb:.1f} KB)")